In [ ]:
import pandas as pd
import time
import os
import psycopg2
from pgvector.psycopg2 import register_vector
from psycopg2.extras import execute_values
import boto3
from sentence_transformers import SentenceTransformer
import numpy as np


In [ ]:
start_time = time.time()
s3 = boto3.client('s3')
os.makedirs('/tmp/model', exist_ok=True)
bucket = 'reelsense-dataset-mlops-carlos'
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=bucket, Prefix='model/reelsense-movie-embeddings'):
    for obj in page.get('Contents', []):
        local_path = '/tmp/' + obj['Key']
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        s3.download_file(bucket, obj['Key'], local_path)

model_path = '/tmp/model/reelsense-movie-embeddings'
print("Loading model...")
model = SentenceTransformer(model_path)

print("loading dataset...")
s3.download_file('reelsense-dataset-mlops-carlos', 'data/movies_enriched.csv', '/tmp/movies_enriched.csv')
movies = pd.read_csv('/tmp/movies_enriched.csv')
columnas = [
        "name",
        "description",
        "tagline",
        "main_actors",
        "main_directors",
        "main_genres",
        "main_themes",
    ]
movies[columnas] = movies[columnas].fillna("")

movies["meta_texto"] = movies[
        ["name", "main_actors", "main_directors", "main_genres", "main_themes"]
    ].agg(" ".join, axis=1)
movies["plot_texto"] = movies[["description", "tagline"]].agg(" ".join, axis=1)





In [ ]:
print("Generating embeddings...")
meta_embeddings = model.encode(
        movies["meta_texto"].tolist(),
        show_progress_bar=True,
        batch_size=512,        # push it higher on CPU
        convert_to_numpy=True  # skip any tensor overhead
    )
plot_embeddings = model.encode(movies["plot_texto"].tolist(),
        show_progress_bar=True, 
        batch_size=512, 
        convert_to_numpy=True
    )


In [ ]:

conn = psycopg2.connect(host='**********',
    port=5432,
    dbname='postgres',
    user='ReelSensepostgres',
    password='******')
register_vector(conn)

cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
cur.execute("DROP TABLE IF EXISTS movie_embeddings;")
cur.execute("""
    CREATE TABLE movie_embeddings (
        id SERIAL PRIMARY KEY,
        movie_id INT,
        name TEXT,
        meta_embedding VECTOR(384),
        plot_embedding VECTOR(384),
        poster_link TEXT
    );
""")
    
rows = list(zip(movies['id'].tolist(),
                    movies['name'].tolist(),
                    [np.array(e) for e in meta_embeddings],
                    [np.array(e) for e in plot_embeddings],
                    movies['link'].tolist()))



execute_values(
    cur,
    "INSERT INTO movie_embeddings (movie_id, name, meta_embedding, plot_embedding, poster_link) VALUES %s",
    rows,
    page_size=1000
)
conn.commit()
cur.close()
conn.close()
    
    # 4. Cerramos el log de rendimiento


In [ ]:
end_time = time.time()
    
print("¡Catálogo vectorizado y trackeado en MLflow con éxito!")
